In [2]:
import heapq

class PuzzleNode:
    def __init__(self, state, g_score, h_score, parent=None, move=""):
        self.state = state        # Tuple of 9 integers representing the 3x3 board
        self.g_score = g_score    # Path cost from start to current node
        self.h_score = h_score    # Heuristic cost (Manhattan Distance)
        self.f_score = g_score + h_score
        self.parent = parent      # Link back to parent node for path reconstruction
        self.move = move          # Direction taken to arrive at this state

    # Priority queue sorting criteria based on f_score
    def __lt__(self, other):
        return self.f_score < other.f_score

def get_manhattan_distance(state, goal_state):
    """Calculates the sum of Manhattan distances of misplaced tiles."""
    distance = 0
    for i in range(9):
        tile = state[i]
        if tile != 0:  # Skip the empty space
            goal_idx = goal_state.index(tile)
            # Convert 1D index to 2D coordinates (row, col)
            curr_row, curr_col = i // 3, i % 3
            goal_row, goal_col = goal_idx // 3, goal_idx % 3
            distance += abs(curr_row - goal_row) + abs(curr_col - goal_col)
    return distance

def get_neighbors(state):
    """Generates valid next states by shifting the empty space (0)."""
    neighbors = []
    blank_idx = state.index(0)
    row, col = blank_idx // 3, blank_idx % 3

    # Define directional moves: (row change, col change, Move Name)
    moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

    for dr, dc, move_name in moves:
        new_row, new_col = row + dr, col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_blank_idx = new_row * 3 + new_col
            # Swap empty tile with target neighbor
            new_state = list(state)
            new_state[blank_idx], new_state[new_blank_idx] = new_state[new_blank_idx], new_state[blank_idx]
            neighbors.append((tuple(new_state), move_name))

    return neighbors

def is_solvable(state, goal_state):
    """
    Checks if a configuration is solvable by tracking total inversions.
    An inversion is when a tile with a higher number precedes a lower number.
    The puzzle is solvable if initial and goal state inversion counts share the same parity.
    """
    def count_inversions(arr):
        flat = [tile for tile in arr if tile != 0]
        inversions = 0
        for i in range(len(flat)):
            for j in range(i + 1, len(flat)):
                if flat[i] > flat[j]:
                    inversions += 1
        return inversions

    return count_inversions(state) % 2 == count_inversions(goal_state) % 2

def solve_a_star(start_state, goal_state):
    """Finds the optimal path from start_state to goal_state using A* Search."""
    if not is_solvable(start_state, goal_state):
        return None, "The puzzle configuration is NOT solvable."

    # Open set tracking discovered nodes using priority queue heap structures
    h_start = get_manhattan_distance(start_state, goal_state)
    start_node = PuzzleNode(start_state, 0, h_start)

    open_list = [start_node]
    # Dictionary tracking the lowest g_score evaluated for each unique board state
    closed_dict = {start_state: 0}

    while open_list:
        current_node = heapq.heappop(open_list)

        if current_node.state == goal_state:
            # Reconstruct the sequence path steps backwards
            path = []
            curr = current_node
            while curr.parent:
                path.append((curr.move, curr.state))
                curr = curr.parent
            path.reverse()
            return path, "Solved successfully!"

        for neighbor_state, move in get_neighbors(current_node.state):
            g_score = current_node.g_score + 1

            # If state has not been seen or a shorter path to it has been found
            if neighbor_state not in closed_dict or g_score < closed_dict[neighbor_state]:
                closed_dict[neighbor_state] = g_score
                h_score = get_manhattan_distance(neighbor_state, goal_state)
                neighbor_node = PuzzleNode(neighbor_state, g_score, h_score, current_node, move)
                heapq.heappush(open_list, neighbor_node)

    return None, "No solution found."

def print_board(state):
    """Helper method to format the 1D representation into a 3x3 console grid."""
    for i in range(0, 9, 3):
        print(f" {state[i]} {state[i+1]} {state[i+2]} ")
    print("-" * 11)

def parse_input_to_tuple():
    """Takes user console strings and converts them to standard structure format arrays."""
    print("Enter the 9 numbers row-by-row separated by spaces (use '0' for the blank space):")
    state = []
    for i in range(3):
        while True:
            try:
                row = list(map(int, input(f"Row {i+1}: ").strip().split()))
                if len(row) == 3:
                    state.extend(row)
                    break
                print("Error: Please enter exactly 3 numbers.")
            except ValueError:
                print("Error: Invalid characters. Enter integers only.")
    return tuple(state)

# --- Execution ---
if __name__ == "__main__":
    print("--- 8-PUZZLE SOLVER (A* ALGORITHM) ---")
    print("\n[STEP 1] Enter the INITIAL State:")
    initial_state = parse_input_to_tuple()

    print("\n[STEP 2] Enter the GOAL State:")
    goal_state = parse_input_to_tuple()

    print("\nCalculating solution using A*...")
    solution_path, status_message = solve_a_star(initial_state, goal_state)

    print(f"\nStatus: {status_message}")
    if solution_path is not None:
        print(f"Total Moves Needed: {len(solution_path)}")
        print("\nInitial Configuration:")
        print_board(initial_state)

        for step_idx, (move, state) in enumerate(solution_path, 1):
            print(f"Step {step_idx}: Move blank tile '{move}'")
            print_board(state)


--- 8-PUZZLE SOLVER (A* ALGORITHM) ---

[STEP 1] Enter the INITIAL State:
Enter the 9 numbers row-by-row separated by spaces (use '0' for the blank space):
Row 1: 2 8 3
Row 2: 1 6 4
Row 3: 7 0 5

[STEP 2] Enter the GOAL State:
Enter the 9 numbers row-by-row separated by spaces (use '0' for the blank space):
Row 1: 1 2 3
Row 2: 8 0 4
Row 3: 7 6 5

Calculating solution using A*...

Status: Solved successfully!
Total Moves Needed: 5

Initial Configuration:
 2 8 3 
 1 6 4 
 7 0 5 
-----------
Step 1: Move blank tile 'Up'
 2 8 3 
 1 0 4 
 7 6 5 
-----------
Step 2: Move blank tile 'Up'
 2 0 3 
 1 8 4 
 7 6 5 
-----------
Step 3: Move blank tile 'Left'
 0 2 3 
 1 8 4 
 7 6 5 
-----------
Step 4: Move blank tile 'Down'
 1 2 3 
 0 8 4 
 7 6 5 
-----------
Step 5: Move blank tile 'Right'
 1 2 3 
 8 0 4 
 7 6 5 
-----------
